In [1]:
import torch
from typing import cast, Optional, Sized
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset
import time
import os
from tqdm import tqdm
from torch import accelerator, Generator
import torch.nn.functional as F
import cv2
import shutil
import numpy as np
from pathlib import Path
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedKFold, train_test_split
import torch.optim as optim
from torch.optim import lr_scheduler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import math
from matplotlib import pyplot as plt
from pytorch_metric_learning.losses import ArcFaceLoss
import timm

In [2]:
import multiprocessing

try:
    multiprocessing.set_start_method("fork", True)
except (RuntimeError, ValueError):
    pass
cv2.setNumThreads(0)

In [3]:
class MLRunner:
    @staticmethod
    def _get_predicts(ys):
        return ys

    def __init__(
        self,
        device=torch.device("cpu"),
        model=None,
        opt=None,
        criterion=None,
        dataset=None,
        seed=42,
        *args,
        **kwargs,
    ):
        self.device = device
        self.model: Optional[nn.Module] = (
            model.to(self.device) if model is not None else None
        )
        self.opt = opt
        self.criterion = (
            criterion
            if not isinstance(criterion, nn.Module)
            else criterion.to(self.device)
        )
        self.dataset = dataset
        self.seed = seed
        if self.dataset is None:
            self.dataloader = None
        else:
            self.dataloader = DataLoader(self.dataset, *args, **kwargs)

    def run(
        self,
        epochs=1,
        max_model_save=0,
        max_epochs_print=-1,
        is_train=False,
        scheduler=None,
        save_dir=None,
        batch_scheduler=False,
        *args,
        **kwargs,
    ):
        assert self.model is not None and self.dataloader is not None
        since = time.time()
        ls, preds = [], []
        save_interval = (
            (epochs // max_model_save) + (0 if epochs % max_model_save == 0 else 1)
            if max_model_save > 0
            else (epochs + 1)
        )
        print_interval = (
            (epochs // max_epochs_print) + (0 if epochs % max_epochs_print == 0 else 1)
            if max_epochs_print > 0
            else (epochs + 1)
        )
        if is_train:
            if scheduler is not None:
                scheduler = scheduler(self.opt, *args, **kwargs)
            self.model.train()
            if isinstance(self.criterion, nn.Module):
                self.criterion.train()
        else:
            self.model.eval()
            if isinstance(self.criterion, nn.Module):
                self.criterion.eval()
        if save_dir is not None:
            os.makedirs(save_dir, exist_ok=True)
        for ep in range(epochs):
            ls.append(0)
            if not is_train:
                preds.append([])
            for batch in tqdm(self.dataloader, desc=f"Epoch {ep+1}/{epochs}"):
                x, y = batch[0].to(self.device), None
                if len(batch) >= 2:
                    y = batch[1].to(self.device)
                else:
                    ls[-1] = None
                if is_train:
                    self.opt.zero_grad()
                with torch.set_grad_enabled(is_train):
                    outputs = self.model(x)
                    if ls[-1] is not None:
                        l = self.criterion(outputs, y)
                    if is_train:
                        l.backward()
                        self.opt.step()
                        if scheduler is not None and batch_scheduler:
                            scheduler.step()
                    else:
                        ps = self._get_predicts(outputs).detach().cpu()
                        preds[-1].append(ps)
                if ls[-1] is not None:
                    ls[-1] += l.item() * x.size(0)
            if is_train:
                if save_dir is not None and (
                    (max_model_save < 0)
                    or (epochs <= max_model_save)
                    or ((ep + 1) % save_interval == 0)
                ):
                    torch.save(
                        self.model.state_dict(),
                        os.path.join(save_dir, f"model_epoch{ep+1}.pth"),
                    )
                    if isinstance(self.criterion, nn.Module):
                        torch.save(
                            self.criterion.state_dict(),
                            os.path.join(save_dir, f"criterion_epoch{ep+1}.pth"),
                        )
            else:
                preds[-1] = torch.cat(preds[-1])
            if scheduler is not None and not batch_scheduler:
                scheduler.step()
            if ls[-1] is not None:
                ls[-1] /= len(cast(Sized, cast(object, self.dataloader.dataset)))
                if (
                    (max_epochs_print < 0)
                    or (epochs <= max_epochs_print)
                    or ((ep + 1) % print_interval == 0)
                ):
                    print(f"Epoch {ep+1}/{epochs}, Loss: {ls[-1]:.6f}")
        if save_dir is not None:
            torch.save(
                self.model.state_dict(), os.path.join(save_dir, f"model_final.pth")
            )
            if isinstance(self.criterion, nn.Module):
                torch.save(
                    self.criterion.state_dict(),
                    os.path.join(save_dir, f"criterion_final.pth"),
                )
        time_elapsed = time.time() - since
        if is_train:
            return ls, time_elapsed
        else:
            return ls, preds, time_elapsed

    def set_criterion(self, criterion):
        self.criterion = (
            criterion
            if not isinstance(criterion, nn.Module)
            else criterion.to(self.device)
        )
        return self.criterion

    def set_dataloader(self, *args, **kwargs):
        self.dataloader = DataLoader(self.dataset, *args, **kwargs)
        return self.dataloader

    def set_dataset(self, dataset):
        self.dataset = dataset
        return self.dataset

    def set_model(self, model):
        self.model = model.to(self.device)
        return self.model

    def set_opt(self, opt, *args, **kwargs):
        assert self.model is not None
        params = list(filter(lambda p: p.requires_grad, self.model.parameters()))
        if isinstance(self.criterion, nn.Module):
            params += list(
                filter(lambda p: p.requires_grad, self.criterion.parameters())
            )
        self.opt = opt(params, *args, **kwargs)
        return self.opt

In [4]:
class ClassificationRunner(MLRunner):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _get_predicts(self, ys):
        ys = ys.cpu()
        mask = ys == torch.max(ys, 1, True).values
        ids = torch.where(mask)[1]
        cnts = mask.sum(1)
        offsets = (
            torch.randint(
                0,
                ys.size(1),
                (ys.size(0),),
                generator=Generator().manual_seed(self.seed),
            )
            % cnts
        )
        starts = torch.cat((torch.tensor([0]), torch.cumsum(cnts, dim=0)[:-1]))
        return ids[starts + offsets]

In [5]:
class EmbedClassificationRunner(ClassificationRunner):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _get_predicts(self, ys):
        return super()._get_predicts(ys @ F.normalize(self.criterion.W.detach(), dim=0))

In [6]:
def pre_proc(img, size, *args, **kwargs):
    img = cv2.bitwise_not(img)
    img = cv2.resize(img, size, *args, **kwargs)
    return img

In [7]:
def build_dataset_dir(dataset_path, exts, org_path):
    if dataset_path.exists():
        shutil.rmtree(dataset_path)
    dataset_path.mkdir(parents=True)
    for ext in exts:
        for f in org_path.glob(ext):
            label_dir = dataset_path / f.stem.split("_")[0]
            label_dir.mkdir(exist_ok=True)
            (label_dir / f.name).symlink_to(
                os.path.relpath(f.resolve(), start=label_dir.resolve())
            )
    return dataset_path

In [8]:
class Augmentation:
    def __init__(
        self,
        close_ksize=(5, 5),
        min_block_ratio=0.05,
        max_transform=0.1,
        move_ratio=0.25,
        p=0.5,
        seed=42,
    ):
        self.close_ksize = close_ksize
        self.min_block_ratio = min_block_ratio
        self.max_transform = max_transform
        self.move_ratio = move_ratio
        self.p = p
        self.rng = np.random.default_rng(seed)

    def __find(self, img):
        _, b_img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        b_img = cv2.morphologyEx(
            b_img,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_RECT, self.close_ksize),
        )
        n, _, stats, _ = cv2.connectedComponentsWithStats(b_img)
        blocks = []
        for i in range(1, n):
            x, y, w, h, _ = stats[i]
            if (
                h > self.min_block_ratio * img.shape[0]
                and w > self.min_block_ratio * img.shape[1]
            ):
                blocks.append([img[y : y + h, x : x + w], x, y, w, h])
        return sorted(blocks, key=lambda b: (b[1]))

    def __warp(self, block):
        h, w = block.shape
        pts = np.array(
            [[0, 0], [w - 1, 0], [0, h - 1], [w - 1, h - 1]], dtype=np.float32
        )
        mat = cv2.getPerspectiveTransform(
            pts,
            (
                pts
                + self.rng.uniform(-1, 1, (4, 2))
                * self.max_transform
                * np.array([w, h])
            ).astype(np.float32),
        )
        block = cv2.warpPerspective(
            block,
            mat,
            (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
        )
        return block

    def __move(self, blocks, img_h):
        org_xs = [x for _, x, _, _, _ in blocks]
        for i, (_, x, y, w, _) in enumerate(blocks):
            blocks[i][1] = int(
                x
                + (
                    self.move_ratio
                    * self.rng.uniform(-1, 1)
                    * (
                        (x + w / 2) - (org_xs[i - 1] + blocks[i - 1][3] / 2)
                        if i > 0
                        else x + w / 2
                    )
                )
            )
            blocks[i][2] = int(
                y + self.move_ratio * self.rng.uniform(-1, 1) * (img_h / 2)
            )
        return blocks

    def __call__(self, img):
        if self.rng.random() < self.p:
            img_inv = np.asarray(cv2.bitwise_not(img))
            blocks = self.__find(img_inv)
            if blocks:
                img_h, img_w = img.shape
                blocks = [
                    [self.__warp(block), x, y, w, h] for block, x, y, w, h in blocks
                ]
                self.__move(blocks, img_h)
                img_inv = np.zeros_like(img_inv)
                for block, x, y, w, h in blocks:
                    x1, y1 = max(x, 0), max(y, 0)
                    x2, y2 = min(int(x + w), img_w), min(int(y + h), img_h)
                    if x2 > x1 and y2 > y1:
                        img_inv[y1:y2, x1:x2] = np.maximum(
                            img_inv[y1:y2, x1:x2],
                            block[(y1 - y) : (y2 - y), (x1 - x) : (x2 - x)],
                        )
                img = cv2.bitwise_not(img_inv)
        return img

In [9]:
DATA_PATH = Path("data/")
INPUT_SIZE = (256, 256)
DATASET_PATH = DATA_PATH / "dataset/"
EXTS = ("*.png",)
PRE_TRANSFORMS = transforms.Compose(
    [lambda img: pre_proc(img, INPUT_SIZE, cv2.INTER_LINEAR), transforms.ToPILImage()]
)
CV2_LOADER = lambda path: cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
DS = datasets.ImageFolder(
    build_dataset_dir(DATASET_PATH, EXTS, DATA_PATH / "org" / "train/"),
    transforms.Compose([PRE_TRANSFORMS, transforms.ToTensor()]),
    loader=CV2_LOADER,
)
CPU_WORKERS = min(os.cpu_count() or 1, 32)
SEED = 42
MEAN, STD, N = 0, 0, 0
for imgs, _ in DataLoader(DS, batch_size=1000, num_workers=CPU_WORKERS):
    MEAN += imgs.sum().item()
    STD += (imgs**2).sum().item()
    N += imgs.numel()
MEAN = MEAN / N
STD = (STD / N - MEAN**2) ** 0.5
TRAIN_TRANSFORMS = transforms.Compose(
    [
        Augmentation((5, 5), 0.05, 0.15, 0.5, 0.75, SEED),
        PRE_TRANSFORMS,
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
VAL_TRANSFORMS = transforms.Compose(
    [
        PRE_TRANSFORMS,
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
FULL_TRAIN = datasets.ImageFolder(DATASET_PATH, TRAIN_TRANSFORMS, loader=CV2_LOADER)
FULL_VAL = datasets.ImageFolder(DATASET_PATH, VAL_TRANSFORMS, loader=CV2_LOADER)

In [10]:
def kfold_split(k, seed=42):
    skf = StratifiedKFold(k, shuffle=True, random_state=seed)
    targets = np.array(FULL_TRAIN.targets)
    for train_idx, val_idx in skf.split(np.zeros(len(targets)), targets):
        yield Subset(FULL_TRAIN, train_idx), Subset(FULL_VAL, val_idx)

In [ ]:
BATCH_SIZE = 52
# BATCH_SIZE = 512
DEVICE = (
    cast(torch.device, accelerator.current_accelerator())
    if accelerator.is_available()
    else torch.device("cpu")
)
OPT = optim.Adam
OPT_PARAMS = {"lr": 1e-4, "weight_decay": 5e-3}
SCHE_PARAMS = {
    "max_lr": 5e-4,
    "pct_start": 0.2,
    "div_factor": 10,
    "final_div_factor": 300,
}
# EPOCHS = 1000
EPOCHS = 3
MAX_MODEL_SAVE = 5
MAX_EPOCHS_PRINT = 100
SCHEDULER = lr_scheduler.OneCycleLR
OUT_DIR = "outputs"
MODEL_DIR = "models"
CLS_NAMES = DS.classes

In [12]:
def get_score(y, y_):
    acc, f1, prec, rec = (
        accuracy_score(y, y_),
        f1_score(y, y_, average="macro", zero_division=0),
        precision_score(y, y_, average="macro", zero_division=0),
        recall_score(y, y_, average="macro", zero_division=0),
    )
    print(
        f"Accuracy: {100*acc:.4f}%, F1 Score: {100*f1:.4f}%, Precision: {100*prec:.4f}%, Recall: {100*rec:.4f}%"
    )
    return acc, f1, prec, rec

In [13]:
def run_model(
    runner,
    criterion_loader,
    train_set,
    model_builder,
    name,
    val_set,
    res_file=None,
    labels=None,
    *args,
    **kwargs,
):
    runner.set_criterion(criterion_loader())
    train_l = None
    if train_set is not None:
        runner.set_dataset(train_set)
        runner.set_dataloader(
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=CPU_WORKERS,
            pin_memory=DEVICE.type != "mps",
            persistent_workers=True,
        )
        runner.set_model(model_builder(*args, **kwargs))
        runner.set_opt(OPT, **OPT_PARAMS)
        SCHE_PARAMS["total_steps"] = math.ceil(len(train_set) / BATCH_SIZE) * EPOCHS
        train_l, _ = runner.run(
            EPOCHS,
            MAX_MODEL_SAVE,
            MAX_EPOCHS_PRINT,
            True,
            SCHEDULER,
            f"{OUT_DIR}/{MODEL_DIR}/{name}",
            True,
            **SCHE_PARAMS,
        )
    val_labels, val_preds = None, None
    if val_set is not None:
        if (
            hasattr(val_set, "dataset")
            and hasattr(val_set.dataset, "targets")
            and hasattr(val_set, "indices")
        ):
            val_labels = [
                cast(datasets.ImageFolder, val_set.dataset).targets[i]
                for i in val_set.indices
            ]
        runner.set_dataset(val_set)
        runner.set_dataloader(
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=CPU_WORKERS,
            pin_memory=DEVICE.type != "mps",
            persistent_workers=True,
        )
        if train_set is None:
            runner.set_model(model_builder(*args, **kwargs))
        _, val_preds, _ = runner.run()
        val_preds = val_preds[-1].numpy()
        if res_file is not None and labels is not None:
            with open(res_file, "w") as f:
                for p, path in zip(val_preds, labels):
                    f.write(f"{path.stem}\t{int(CLS_NAMES[p])}\n")
    if val_labels is not None:
        return train_l, get_score(val_labels, val_preds)
    else:
        return train_l

In [14]:
EMBED_DIM = 384
CRIT_PARAMS = {"num_classes": len(CLS_NAMES), "embedding_size": EMBED_DIM}
IMG_DIR = "imgs"

In [15]:
def plot_loss(ls, name, save_file=None):
    for label, l in ls:
        plt.plot(l, label=label)
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.title(f"{name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    if save_file:
        plt.savefig(save_file, bbox_inches="tight")
    else:
        plt.show()
    plt.close()

In [16]:
def run_model_kf(runner, k, model_builder, name, *args, **kwargs):
    train_ls, accs, f1s, precs, recs = [], [], [], [], []
    for fold, (train_set, val_set) in enumerate(kfold_split(k, SEED)):
        print(f"Fold {fold+1}/{k}")
        l, (acc, f1, prec, rec) = run_model(
            runner,
            lambda: ArcFaceLoss(**CRIT_PARAMS),
            train_set,
            model_builder,
            f"{name}_fold{fold+1}",
            val_set,
            *args,
            **kwargs,
        )
        train_ls.append((f"fold {fold+1}", l))
        accs.append(acc)
        f1s.append(f1)
        precs.append(prec)
        recs.append(rec)
    os.makedirs(f"{OUT_DIR}/{IMG_DIR}", exist_ok=True)
    plot_loss(train_ls, name, save_file=f"{OUT_DIR}/{IMG_DIR}/loss_{name}_kf.svg")
    print(f"Average Accuracy: {100 * sum(accs) / len(accs):.4f}%")
    print(f"Average F1-Score: {100 * sum(f1s) / len(f1s):.4f}%")
    print(f"Average Precision: {100 * sum(precs) / len(precs):.4f}%")
    print(f"Average Recall: {100 * sum(recs) / len(recs):.4f}%")

In [17]:
class Norm(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.args = args
        self.kwargs = kwargs

    def forward(self, x):
        return F.normalize(x, *self.args, **self.kwargs)

In [18]:
def build_model(backbone_name, in_chans, embed_dim, drop_out=0.5, *args, **kwargs):
    model = timm.create_model(
        backbone_name, in_chans=in_chans, num_classes=0, *args, **kwargs
    )
    model = nn.Sequential(
        model,
        nn.Dropout(drop_out),
        nn.Linear(getattr(model, "num_features"), embed_dim, bias=False),
        nn.BatchNorm1d(embed_dim),
        Norm(),
    )
    return model

In [19]:
def build_test_set(test_path, exts):
    paths = []
    for ext in exts:
        paths.extend(test_path.glob(ext))
    dataset = [VAL_TRANSFORMS(CV2_LOADER(p)) for p in paths]
    dataset = TensorDataset(torch.stack(dataset))
    return dataset, paths

In [20]:
def load_model(model_builder, model_file, *args, **kwargs):
    model = model_builder(*args, **kwargs)
    model.load_state_dict(torch.load(model_file, "cpu"))
    return model

In [21]:
RUNNER = EmbedClassificationRunner(device=DEVICE)
MODEL_NAME = "ConvNeXtTiny_ArcFace"
MODEL_PARAMS = {
    "backbone_name": "convnext_tiny",
    "in_chans": 1,
    "embed_dim": EMBED_DIM,
    "drop_out": 0.5,
    "pretrained": True,
    "drop_path_rate": 0.1,
}

In [22]:
run_model_kf(RUNNER, 3, build_model, MODEL_NAME, **MODEL_PARAMS)

Fold 1/3


Epoch 1/3: 100%|██████████| 11/11 [00:35<00:00,  3.20s/it]


Epoch 1/3, Loss: 39.773849


Epoch 2/3: 100%|██████████| 11/11 [00:32<00:00,  2.96s/it]


Epoch 2/3, Loss: 39.260931


...

Epoch 2/3, Loss: 39.507898


Epoch 3/3: 100%|██████████| 11/11 [00:57<00:00,  5.21s/it]


Epoch 3/3, Loss: 39.310063


Epoch 1/1: 100%|██████████| 6/6 [00:07<00:00,  1.21s/it]

Epoch 1/1, Loss: 40.924763
Accuracy: 1.2012%, F1 Score: 0.0237%, Precision: 0.0120%, Recall: 1.0000%
Average Accuracy: 1.1002%
Average F1-Score: 0.0218%
Average Precision: 0.0110%
Average Recall: 1.0000%


In [23]:
TARGETS = np.array(DS.targets)
TRAIN_SET, VAL_SET = train_test_split(
    np.arange(len(TARGETS)), test_size=1 - 0.8, random_state=SEED, stratify=TARGETS
)
TRAIN_SET = Subset(FULL_TRAIN, TRAIN_SET)
VAL_SET = Subset(FULL_VAL, VAL_SET)

In [24]:
loss, _ = run_model(
    RUNNER,
    lambda: ArcFaceLoss(**CRIT_PARAMS),
    TRAIN_SET,
    build_model,
    MODEL_NAME,
    VAL_SET,
    **MODEL_PARAMS,
)
os.makedirs(f"{OUT_DIR}/{IMG_DIR}", exist_ok=True)
plot_loss(
    [("train", loss)],
    MODEL_NAME,
    save_file=f"{OUT_DIR}/{IMG_DIR}/loss_{MODEL_NAME}.svg",
)

Epoch 1/3: 100%|██████████| 13/13 [01:02<00:00,  4.77s/it]


Epoch 1/3, Loss: 39.358833


Epoch 2/3: 100%|██████████| 13/13 [01:06<00:00,  5.14s/it]


Epoch 2/3, Loss: 39.416529


Epoch 3/3: 100%|██████████| 13/13 [01:11<00:00,  5.52s/it]


Epoch 3/3, Loss: 39.185377


Epoch 1/1: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]

Epoch 1/1, Loss: 37.750074
Accuracy: 1.0000%, F1 Score: 0.0198%, Precision: 0.0100%, Recall: 1.0000%


In [25]:
loss = run_model(
    RUNNER,
    lambda: ArcFaceLoss(**CRIT_PARAMS),
    FULL_TRAIN,
    build_model,
    MODEL_NAME + "_full",
    None,
    **MODEL_PARAMS,
)
os.makedirs(f"{OUT_DIR}/{IMG_DIR}", exist_ok=True)
plot_loss(
    [("train", loss)],
    MODEL_NAME + "_full",
    save_file=f"{OUT_DIR}/{IMG_DIR}/loss_{MODEL_NAME}_full.svg",
)

Epoch 1/3: 100%|██████████| 16/16 [01:17<00:00,  4.86s/it]


Epoch 1/3, Loss: 39.617319


Epoch 2/3: 100%|██████████| 16/16 [01:24<00:00,  5.30s/it]


Epoch 2/3, Loss: 39.088234


Epoch 3/3: 100%|██████████| 16/16 [01:27<00:00,  5.48s/it]


Epoch 3/3, Loss: 38.742888


In [26]:
VAL_SET, IMG_NAMES = build_test_set(Path("data/org/test/"), EXTS)
RES_FILE = f"{OUT_DIR}/{MODEL_NAME}_results.txt"

In [27]:
run_model(
    RUNNER,
    lambda: load_model(
        ArcFaceLoss,
        f"{OUT_DIR}/{MODEL_DIR}/{MODEL_NAME}_full/criterion_final.pth",
        **CRIT_PARAMS,
    ),
    None,
    lambda *args, **kwargs: load_model(
        build_model,
        f"{OUT_DIR}/{MODEL_DIR}/{MODEL_NAME}_full/model_final.pth",
        *args,
        **kwargs,
    ),
    MODEL_NAME,
    VAL_SET,
    RES_FILE,
    IMG_NAMES,
    **MODEL_PARAMS,
)

Epoch 1/1: 100%|██████████| 157/157 [02:41<00:00,  1.03s/it]


# TEST


In [28]:
def load_results(res_path):
    results = {}
    with open(res_path, "r") as f:
        for line in f:
            name, label = line.strip().split("\t")
            results[name] = label
    return results

In [29]:
pred_res, true_res = load_results(RES_FILE), load_results(
    DATA_PATH / "org" / "truth.txt"
)
test_y, test_y_ = [], []
for img_name in true_res:
    test_y.append(true_res[img_name])
    test_y_.append(pred_res[img_name])
get_score(test_y, test_y_)

Accuracy: 1.1800%, F1 Score: 0.0233%, Precision: 0.0118%, Recall: 1.0000%


(0.0118, 0.0002332476774066021, 0.000118, 0.01)